# Simple RAG (Retrieval-Augmented Generation) System

This notebook builds a RAG pipeline for question-answering over your own
documents (PDF / TXT), following the architecture:

**Document Ingestion → Text Chunking → Embedding Creation → Vector Database
→ Query Processing → Context Retrieval → Answer Generation**

Retrieval uses TF-IDF + cosine similarity (scikit-learn), so the whole
notebook runs offline with no model downloads. Answer generation defaults to
an "extractive" mode (returns the most relevant passages) and can optionally
call the Anthropic or OpenAI API for a synthesized answer — see the Answer
Generation section.

## Setup

In [1]:
!pip install -q pypdf scikit-learn numpy

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import os
import glob
from dataclasses import dataclass, field
from typing import List, Tuple

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader

## 1. Document Ingestion

Load raw text out of PDFs or plain-text files. Point `DOCS_FOLDER` at a
folder containing your own PDFs, notes, resume, research papers, etc.

In [3]:
class DocumentLoader:
    """Loads raw text out of PDFs and plain-text files."""

    @staticmethod
    def load_file(path: str) -> str:
        if path.lower().endswith(".pdf"):
            reader = PdfReader(path)
            pages = [page.extract_text() or "" for page in reader.pages]
            return "\n".join(pages)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    @staticmethod
    def load_folder(folder: str, patterns=("*.pdf", "*.txt", "*.md")) -> List[Tuple[str, str]]:
        """Returns a list of (filename, text) for every matching file in folder."""
        docs = []
        for pattern in patterns:
            for path in sorted(glob.glob(os.path.join(folder, pattern))):
                text = DocumentLoader.load_file(path)
                if text.strip():
                    docs.append((os.path.basename(path), text))
        return docs

In [4]:
DOCS_FOLDER = "documents"  # <- put your own PDFs/TXT files in this folder

docs = DocumentLoader.load_folder(DOCS_FOLDER)
print(f"Loaded {len(docs)} document(s):")
for name, text in docs:
    print(f"  - {name}  ({len(text)} characters)")

Loaded 1 document(s):
  - Week7_Project.pdf  (4198 characters)


## 2. Text Chunking

Split each document into overlapping word-based chunks so retrieval can
work on small, focused pieces of text instead of whole documents.

In [5]:
@dataclass
class Chunk:
    text: str
    source: str = ""


class TextChunker:
    """Splits text into overlapping word-based chunks."""

    def __init__(self, chunk_size: int = 200, overlap: int = 40):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def split(self, text: str, source: str = "") -> List[Chunk]:
        words = text.split()
        chunks = []
        step = max(1, self.chunk_size - self.overlap)
        for start in range(0, len(words), step):
            piece = words[start:start + self.chunk_size]
            if not piece:
                continue
            chunks.append(Chunk(text=" ".join(piece), source=source))
            if start + self.chunk_size >= len(words):
                break
        return chunks

In [6]:
chunker = TextChunker(chunk_size=200, overlap=40)

all_chunks = []
for name, text in docs:
    all_chunks.extend(chunker.split(text, source=name))

print(f"Created {len(all_chunks)} chunks total.\n")
print("Sample chunk:")
print(all_chunks[0].text[:400], "...")

Created 3 chunks total.

Sample chunk:
Document Question Answering System (RAG) Dataset: Simple Beginner Dataset (Easiest) Use your own PDFs: ● Notes ● Resume ● Research papers ● Books ● RAG is meant for custom/private data Or Try this Hugging Face Dataset Reference:Github Link Overview This project implements a Retrieval-Augmented Generation (RAG) system that answers questions based on custom documents. Instead of relying only on a la ...


## 3 & 4. Embedding Creation + Vector Database

Convert each chunk into a vector (TF-IDF here) and store all vectors so we
can search them by similarity. This plays the role of the "vector database"
in the architecture, kept in-memory for simplicity.

In [7]:
class VectorStore:
    """TF-IDF backed vector store with cosine-similarity search."""

    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words="english", max_features=8000)
        self.matrix = None
        self.chunks: List[Chunk] = []

    def build(self, chunks: List[Chunk]):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        self.matrix = self.vectorizer.fit_transform(texts)

    def search(self, query: str, top_k: int = 3) -> List[Tuple[Chunk, float]]:
        if self.matrix is None or not self.chunks:
            return []
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.matrix)[0]
        ranked = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[i], float(sims[i])) for i in ranked if sims[i] > 0]

In [8]:
store = VectorStore()
store.build(all_chunks)
print(f"Vector store built: {store.matrix.shape[0]} chunks x {store.matrix.shape[1]} TF-IDF features")

Vector store built: 3 chunks x 176 TF-IDF features


## 5 & 6. Query Processing + Context Retrieval

Turn the user's question into a vector in the same space, then retrieve the
most similar chunks.

In [9]:
test_query = "What are the stages of the RAG system architecture?"
results = store.search(test_query, top_k=3)

for chunk, score in results:
    print(f"score={score:.3f}  source={chunk.source}")
    print(chunk.text[:250], "...\n")

score=0.163  source=Week7_Project.pdf
Document Question Answering System (RAG) Dataset: Simple Beginner Dataset (Easiest) Use your own PDFs: ● Notes ● Resume ● Research papers ● Books ● RAG is meant for custom/private data Or Try this Hugging Face Dataset Reference:Github Link Overview T ...

score=0.094  source=Week7_Project.pdf
content is added to the model’s input to provide context for answering. 3. Generation A language model generates the final answer using the retrieved context, ensuring responses are grounded in actual data. System Architecture The pipeline consists o ...

score=0.068  source=Week7_Project.pdf
knowledge. Components Used ● Embedding model for converting text into vectors ● Vector store for similarity search ● Language model for generating answers Workflow 1. Load and preprocess documents 2. Split text into chunks 3. Convert chunks into embe ...



## 7. Answer Generation

Turn the retrieved chunks into a final answer.

- `backend="extractive"` — no API key needed; returns the most relevant
  passages directly. Good for verifying the pipeline works end-to-end.
- `backend="anthropic"` / `"openai"` — calls a real LLM to synthesize a
  grounded answer from the retrieved context (requires an API key set as
  an environment variable, and `pip install anthropic` / `pip install openai`).

In [10]:
class AnswerGenerator:
    def __init__(self, backend: str = "extractive", model: str = "claude-sonnet-5"):
        self.backend = backend
        self.model = model
        self.client = None
        if backend == "anthropic":
            import anthropic
            self.client = anthropic.Anthropic()
        elif backend == "openai":
            import openai
            self.client = openai.OpenAI()
        elif backend != "extractive":
            raise ValueError(f"Unknown backend: {backend}")

    def generate(self, question: str, retrieved: List[Tuple[Chunk, float]]) -> str:
        if not retrieved:
            return "I couldn't find anything relevant to that question in the document(s)."

        chunks = [c for c, _ in retrieved]

        if self.backend == "extractive":
            parts = [f"(from {c.source})\n{c.text}" for c in chunks]
            return ("No LLM backend configured, so here are the most relevant "
                    "passages found in the document(s):\n\n" + "\n\n---\n\n".join(parts))

        context = "\n\n".join(f"[Source: {c.source}]\n{c.text}" for c in chunks)
        prompt = (
            "Answer the question using ONLY the context below. "
            "If the answer is not contained in the context, say so explicitly.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
        )

        if self.backend == "anthropic":
            resp = self.client.messages.create(
                model=self.model, max_tokens=500,
                messages=[{"role": "user", "content": prompt}],
            )
            return resp.content[0].text

        if self.backend == "openai":
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
            )
            return resp.choices[0].message.content

In [11]:
generator = AnswerGenerator(backend="extractive")  # switch to "anthropic" or "openai" for real generation
answer = generator.generate(test_query, results)
print(answer)

No LLM backend configured, so here are the most relevant passages found in the document(s):

(from Week7_Project.pdf)
Document Question Answering System (RAG) Dataset: Simple Beginner Dataset (Easiest) Use your own PDFs: ● Notes ● Resume ● Research papers ● Books ● RAG is meant for custom/private data Or Try this Hugging Face Dataset Reference:Github Link Overview This project implements a Retrieval-Augmented Generation (RAG) system that answers questions based on custom documents. Instead of relying only on a language model’s internal knowledge, the system retrieves relevant information from documents and then generates answers grounded in that information. This improves factual accuracy and allows question answering over private or domain-specific data. Objectives ● Understand the concept of Retrieval-Augmented Generation (RAG) ● Build a pipeline combining retrieval and generation ● Enable question answering over custom documents such as PDFs or text files ● Learn how modern AI syste

## Full Pipeline

Wraps every stage above into one reusable class so you can ingest a folder
and ask questions in two lines.

In [12]:
@dataclass
class RAGPipeline:
    chunk_size: int = 200
    overlap: int = 40
    top_k: int = 3
    backend: str = "extractive"
    model: str = "claude-sonnet-5"

    chunker: TextChunker = field(init=False)
    store: VectorStore = field(init=False)
    generator: AnswerGenerator = field(init=False)

    def __post_init__(self):
        self.chunker = TextChunker(self.chunk_size, self.overlap)
        self.store = VectorStore()
        self.generator = AnswerGenerator(self.backend, self.model)

    def ingest_folder(self, folder: str):
        docs = DocumentLoader.load_folder(folder)
        if not docs:
            raise FileNotFoundError(f"No .pdf/.txt/.md files found in {folder}")
        chunks = []
        for name, text in docs:
            chunks.extend(self.chunker.split(text, source=name))
        self.store.build(chunks)
        return len(docs), len(chunks)

    def ask(self, question: str) -> dict:
        retrieved = self.store.search(question, top_k=self.top_k)
        answer = self.generator.generate(question, retrieved)
        return {
            "question": question,
            "answer": answer,
            "sources": [{"source": c.source, "score": round(s, 3)} for c, s in retrieved],
        }

In [13]:
pipeline = RAGPipeline(backend="extractive")
n_docs, n_chunks = pipeline.ingest_folder(DOCS_FOLDER)
print(f"Ingested {n_docs} document(s) -> {n_chunks} chunks\n")

for q in [
    "What is the main idea of the document?",
    "What is RAG used for?",
]:
    result = pipeline.ask(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer'][:500]}...\n")
    print("Sources:", result["sources"])
    print("-" * 70)

Ingested 1 document(s) -> 3 chunks

Q: What is the main idea of the document?
A: No LLM backend configured, so here are the most relevant passages found in the document(s):

(from Week7_Project.pdf)
knowledge. Components Used ● Embedding model for converting text into vectors ● Vector store for similarity search ● Language model for generating answers Workflow 1. Load and preprocess documents 2. Split text into chunks 3. Convert chunks into embeddings 4. Store embeddings in a vector database 5. Accept user query 6. Retrieve relevant chunks 7. Generate answer using retrieved ...

Sources: [{'source': 'Week7_Project.pdf', 'score': 0.133}, {'source': 'Week7_Project.pdf', 'score': 0.038}, {'source': 'Week7_Project.pdf', 'score': 0.02}]
----------------------------------------------------------------------
Q: What is RAG used for?
A: No LLM backend configured, so here are the most relevant passages found in the document(s):

(from Week7_Project.pdf)
knowledge. Components Used ● Embedding mo

## Try your own question

In [14]:
your_question = "What is the main idea of the document?"  # <- change this
result = pipeline.ask(your_question)
print(result["answer"])

No LLM backend configured, so here are the most relevant passages found in the document(s):

(from Week7_Project.pdf)
knowledge. Components Used ● Embedding model for converting text into vectors ● Vector store for similarity search ● Language model for generating answers Workflow 1. Load and preprocess documents 2. Split text into chunks 3. Convert chunks into embeddings 4. Store embeddings in a vector database 5. Accept user query 6. Retrieve relevant chunks 7. Generate answer using retrieved context Example Flow User Question: “What is the main idea of the document?” System Process: ● Retrieves relevant sections ● Provides them as context ● Generates a concise answer Improvements & Experiments ● Use better chunking strategies ● Try different embedding models ● Improve retrieval using hybrid search (keyword + vector) ● Add re-ranking for better relevance ● Experiment with different language models Key Learnings ● How RAG systems combine retrieval and generation ● Importance of retrie

## Next steps / improvements

- **Neural embeddings**: swap the TF-IDF `VectorStore` for
  `sentence-transformers` (e.g. `all-MiniLM-L6-v2`) + a proper vector index
  (FAISS / Chroma / Pinecone) for semantic (not just keyword) similarity.
- **Hybrid search**: combine TF-IDF keyword scores with neural embedding
  scores.
- **Re-ranking**: re-score the top-k retrieved chunks with a cross-encoder
  before generation.
- **Better chunking**: split on sentence/paragraph boundaries instead of a
  fixed word count.
- **Real generation**: set `backend="anthropic"` or `backend="openai"` (with
  the matching API key exported as an environment variable) to have an LLM
  synthesize the final answer instead of returning raw passages.